<a href="https://colab.research.google.com/github/talhanoor23/generative-ai/blob/main/RAGS/Manual_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install chromadb

In [ ]:
%pip install -U langchain-community
%pip install -U langchain-huggingface
%pip install -U bitsandbytes transformers
%pip install pypdf

In [ ]:
# file: document_pipeline.py

# Document loaders
from langchain_community.document_loaders import PyPDFLoader, TextLoader, UnstructuredPDFLoader

# Text splitting
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Vectorstore
from langchain_community.vectorstores import Chroma

from pathlib import Path

from langchain_huggingface import HuggingFaceEmbeddings



class DocumentPipeline:
    def __init__(self, chunk_size=500, chunk_overlap=50, persist_dir="./chroma_db"):
        self.splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
        self.embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
        self.persist_dir = persist_dir

        # Supported loaders mapping
        self.loaders = {
            ".pdf": PyPDFLoader,
            ".txt": TextLoader,
            # ".docx": UnstructuredWordDocumentLoader,
            # ".html": UnstructuredHTMLLoader,
        }

    def load_docs(self, file_path: str):
        ext = Path(file_path).suffix.lower()

        if ext not in self.loaders:
            raise ValueError(f"Unsupported file type: {ext}")

        loader_class = self.loaders[ext]
        loader = loader_class(file_path)
        docs = loader.load()

        return docs, ext.replace(".", "")  # return docs + doctype string

    def chunk_and_tag(self, docs, doctype: str):
        chunks = self.splitter.split_documents(docs)
        for doc in chunks:
            doc.metadata["doctype"] = doctype
        return chunks

    def build_vectorstore(self, file_paths: list[str]):
        all_chunks = []

        for file in file_paths:
            docs, doctype = self.load_docs(file)
            chunks = self.chunk_and_tag(docs, doctype)
            all_chunks.extend(chunks)

        vectorstore = Chroma.from_documents(all_chunks, embedding=self.embeddings, persist_directory=self.persist_dir)
        return vectorstore


In [ ]:
# file: prompt_pipeline.py
from langchain.prompts import ChatPromptTemplate

class PromptPipeline:
    def __init__(self, base_template: str = None):
        # Default template
        self.template = base_template or """
        You are a helpful assistant.
        Use the provided context to answer the question accurately.

        Context:
        {context}

        Question:
        {query}

        Answer:
        """

    def build_prompt(self, context: str, query: str) -> str:
        prompt = ChatPromptTemplate.from_template(self.template)
        return prompt.format(context=context, query=query)

    def change_template(self, new_template: str):
        """Update the template dynamically (e.g. for summarization, Q&A, etc.)."""
        self.template = new_template


In [ ]:
# file: rag_pipeline.py
from operator import itemgetter
from langchain.schema.runnable import RunnableLambda
from langchain.schema.output_parser import StrOutputParser

class RAGPipeline:
    def __init__(self, llm, persist_dir="./chroma_db"):
        # Sub-pipelines
        self.doc_pipeline = DocumentPipeline(persist_dir=persist_dir)
        self.prompt_pipeline = PromptPipeline()

        # LLM (any langchain-compatible model, e.g., HuggingFace, OpenAI, etc.)
        self.llm = llm

    # --- Helpers ---
    @staticmethod
    def docs_to_context(docs):
        """Convert a list of Documents into a single context string."""
        return "\n\n".join([d.page_content for d in docs])

    # --- Build retriever ---
    def get_retriever(self, files=None):
        if files:  # optional: load & build vectorstore if files are passed
            vectorstore = self.doc_pipeline.build_vectorstore(files)
        else:      # or just load existing chroma db
            from langchain_community.vectorstores import Chroma
            vectorstore = Chroma(
                persist_directory=self.doc_pipeline.persist_dir,
                embedding_function=self.doc_pipeline.embeddings
            )
        return vectorstore.as_retriever()

    # --- Build RAG chain ---
    def build_chain(self, retriever):
        return (
            {
                "context": itemgetter("query") | retriever | RunnableLambda(self.docs_to_context),
                "query": itemgetter("query"),
            }
            | RunnableLambda(lambda x: self.prompt_pipeline.build_prompt(x["context"], x["query"]))
            | self.llm
            | StrOutputParser()
        )

    # --- Run once ---
    async def ainvoke(self, query, retriever):
        chain = self.build_chain(retriever)
        return await chain.ainvoke({"query": query})

    def invoke(self, query, retriever):
        chain = self.build_chain(retriever)
        return chain.invoke({"query": query})


In [ ]:
# # Step 1: Init LLM
# from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
# from langchain.llms import HuggingFacePipeline

# model_id = "tiiuae/falcon-7b-instruct"
# tokenizer = AutoTokenizer.from_pretrained(model_id)
# model = AutoModelForCausalLM.from_pretrained(model_id)

# pipe = pipeline(
#     "text-generation",
#     model=model,
#     tokenizer=tokenizer,
#     max_new_tokens=512,
#     temperature=0.8,
# )

# llm = HuggingFacePipeline(pipeline=pipe)

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain.llms import HuggingFacePipeline

model_id = "tiiuae/falcon-7b-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Load quantized model (4-bit)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",        # automatically selects GPU/CPU
    load_in_4bit=True,        # 4-bit quantization
    torch_dtype="auto"
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.8
)

llm = HuggingFacePipeline(pipeline=pipe)

In [ ]:
# Step 2: Init RAG pipeline
rag = RAGPipeline(llm=llm, persist_dir="./chroma_db")

# Step 3: Get retriever (build from files or load existing)
retriever = rag.get_retriever(files=["_Broken Access Control_ A Comprehensive Guide.pdf"])

# Step 4: Run a query
answer = rag.invoke("What is quantum computing?", retriever)
print("Answer:", answer)

Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.


Answer: Human: 
        You are a helpful assistant.
        Use the provided context to answer the question accurately.

        Context:
        ●  Monitor  logs  and  alert  administrators  of  repeated  unauthorized  attempts.  
 
6.  Secure  Cloud  Storage  &  Databases:  
○  Configure  S3  buckets,  Google  Cloud,  and  Azure  storage  with  proper  
permissions.
 ○  Encrypt  sensitive  data  at  rest  and  in  transit.  
7.  Keep  Software  Updated:  
○  Regularly  update  web  servers,  databases,  and  third-party  libraries .  ○  Use  automated  patch  management  tools  to  apply  updates.

●  View  or  modify  sensitive  user  data.  
 ●  Delete  important  records  or  files.  
 ●  Take  control  of  user  accounts  or  escalate  their  privileges.  
 ●  Access  admin  features  without  proper  authorization.  
 ●  Exploit  system  weaknesses  to  launch  further  attacks.  
 ●  Bypass  security  mechanisms  like  login  pages  or  API  access  controls.  
 
Real-world  c

In [ ]:
answer = rag.invoke("What is Conduct Regular Security Testing?", retriever)
print("Answer:", answer)

Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.


Answer: Human: 
        You are a helpful assistant.
        Use the provided context to answer the question accurately.

        Context:
        4.  Validate  User  Input  &  Requests  
●  Prevent  parameter  tampering  by  using  server-side  authorization  checks.  
 ●  Validate  and  sanitize  input  to  prevent  injection  attacks.  
 ●  Use  cryptographically  signed  tokens  (JWT)  to  prevent  tampering.  
 
5.  Conduct  Regular  Security  Testing  
●  Perform  penetration  testing  to  identify  access  control  flaws.  
 ●  Implement  automated  security  scanning  tools  like  OWASP  ZAP,  Burp  Suite,  or  Nmap .

developers
 
can
 
build
 
safer
 
applications
 
that
 
protect
 
user
 
data
 
and
 
business
 
assets.
 
References  
●  OWASP  Proactive  Controls:  Enforce  Access  Controls  
 ●  OWASP  Testing  Guide:  Authorization  Testing  
 ●  NIST  Guide  to  General  Server  Hardening  
 ●  PortSwigger:  Exploiting  CORS  Misconfiguration  
 ●  OAuth:  Revoking  Acce